In [ ]:
!git clone https://github.com/My-Name-Hung/AI_GEN_IMGAE.git /kaggle/working/AI_GEN
%cd /kaggle/working/AI_GEN

In [ ]:
%cd /kaggle/working/AI_GEN
!pip install "torch>=2.4.0"
!pip install "torchvision>=0.19.0"
!pip install "torchaudio>=2.4.0"
!pip install "diffusers>=0.30.3"
!pip install "transformers>=4.41.2"
!pip install "accelerate>=0.34.2"
!pip install "peft>=0.12.0"
!pip install "safetensors>=0.4.3"
!pip install "huggingface_hub>=0.23.0"
!pip install "open_clip_torch>=2.23.0"
!pip install "Pillow>=10.0.0"
!pip install "opencv-python>=4.8.0"
!pip install "opencv-contrib-python>=4.8.0"
!pip install "scikit-image>=0.22.0"
!pip install "scipy>=1.11.0"
!pip install "fastapi>=0.109.0"
!pip install "uvicorn[standard]>=0.25.0"
!pip install "python-multipart>=0.0.6"
!pip install "aiofiles>=22.1.0"
!pip install "pydantic>=2.5.0"
!pip install "python-dotenv>=1.0.0"
!pip install "numpy>=1.24.0"
!pip install "tqdm>=4.66.0"

In [ ]:
RAW_DATASET = '/kaggle/input/datasets/hng111120/gen-ai-data'  # <-- EDIT THIS
PROCESSED_DATASET = '/kaggle/working/dataset_processed'

import os
if not os.path.exists(RAW_DATASET):
    raise FileNotFoundError('Dataset not found: ' + RAW_DATASET + ' -- please edit this cell!')
else:
    print('Dataset found:', RAW_DATASET)

In [ ]:
!python -m training.data_pipeline.paired_poster_import \
    --input "{RAW_DATASET}" \
    --output "{PROCESSED_DATASET}" \
    --target_size 512 \
    --merge_short_title

In [15]:
import os, subprocess, sys
from pathlib import Path

# Prevent torch._dynamo from stalling on Python 3.12 (Kaggle)
os.environ.setdefault('TORCHDYNAMO_DISABLE', '1')
os.environ.setdefault('TORCH_COMPILE_DISABLE', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

os.chdir('/kaggle/working/AI_GEN')
OUTPUT_DIR = '/kaggle/working/outputs'

# Hotfix trực tiếp trên Kaggle để tránh phải clone/push lại repo:
# RuntimeError: Input type (float) and bias type (c10::Half) should be the same
train_file = Path('/kaggle/working/AI_GEN/training/lora_trainer/train_lora.py')
if train_file.exists():
    src = train_file.read_text(encoding='utf-8')
    old = 'pixel_values = batch["pixel_values"].to(device)'
    new = (
        'vae_dtype = next(vae.parameters()).dtype\n'
        '            pixel_values = batch["pixel_values"].to(device=device, dtype=vae_dtype)'
    )
    if old in src and new not in src:
        src = src.replace(old, new, 1)
        train_file.write_text(src, encoding='utf-8')
        print('Applied Kaggle hotfix: cast pixel_values to VAE dtype')

cmd = [
    sys.executable, '-m', 'training.lora_trainer.train_lora',
    '--model_name', 'stabilityai/sdxl-turbo',
    '--dataset', PROCESSED_DATASET,
    '--output', OUTPUT_DIR + '/lora_poster',
    '--rank', '8',
    '--alpha', '16',
    '--lr', '1e-4',
    '--epochs', '10',
    '--batch_size', '1',
    '--grad_accum', '8',
    '--resolution', '512',
    '--save_steps', '100',
    '--validation_prompt',
    'poster_style, modern movie poster, cinematic lighting, high detail',
]

print('Starting LoRA training...')
print('Dataset:', PROCESSED_DATASET)
print('Output :', OUTPUT_DIR + '/lora_poster')
print('First download may take 5-15 min. Do NOT press Stop.')

r = subprocess.run(cmd, env=os.environ.copy())
if r.returncode != 0:
    print(f'Training failed (exit {r.returncode})')
    raise SystemExit(r.returncode)

print('Training complete!')


Applied Kaggle hotfix: cast pixel_values to VAE dtype
Starting LoRA training...
Dataset: /kaggle/working/dataset_processed
Output : /kaggle/working/outputs/lora_poster
First download may take 5-15 min. Do NOT press Stop.


2026-04-06 04:36:31.652967: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775450191.675631     946 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775450191.683190     946 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775450191.702870     946 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775450191.702918     946 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775450191.702924     946 computation_placer.cc:177] computation placer alr

trainable params: 11,612,160 || all params: 2,579,075,844 || trainable%: 0.4502


/kaggle/working/AI_GEN/training/lora_trainer/train_lora.py:248: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp_ctx():
Epoch 1/10:   3%|▎         | 18/590 [00:28<14:01,  1.47s/it, loss=nan]Token indices sequence length is longer than the specified maximum sequence length for this model (87 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['exhibition of the exhibition of the exhibition of the exhibition']
Token indices sequence length is longer than the specified maximum sequence length for this model (87 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['exhibition of the exhibition of the exhibition of the exhibition']
Epoch 1/10:   6%|▋   

Training complete!


In [ ]:
# Test inference with trained LoRA
import os
import torch
from diffusers import AutoPipelineForText2Image

pipeline = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sdxl-turbo',
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipeline = pipeline.to('cuda')

lora_path = '/outputs/lora_poster/final/unet_lora/pytorch_lora_weights.safetensors'
if os.path.exists(lora_path):
    pipeline.load_lora_weights(lora_path)
    print('LoRA weights loaded.')

prompt = 'logo blue texas'
image = pipeline(prompt, num_inference_steps=4, guidance_scale=1.0).images[0]
image.save('outputs/test_output.png')
print('Saved to /kaggle/working/outputs/test_output.png')


2026-04-06 07:11:10.851757: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775459470.874707      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775459470.882185      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775459470.902344      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775459470.902365      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775459470.902368      55 computation_placer.cc:177] computation placer alr

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

text_encoder_2/model.safetensors:   0%|          | 0.00/2.78G [00:00<?, ?B/s]

text_encoder/model.safetensors:   0%|          | 0.00/492M [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/10.3G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

Saved to /kaggle/working/outputs/test_output.png


In [17]:
!ls -lh /kaggle/working/outputs/lora_poster/final/unet_lora/

total 45M
-rw-r--r-- 1 root root  758 Apr  6 07:07 adapter_config.json
-rw-r--r-- 1 root root  45M Apr  6 07:07 adapter_model.safetensors
-rw-r--r-- 1 root root 5.1K Apr  6 07:07 README.md


In [18]:
# Package LoRA weights as ZIP for easy download
!cd /kaggle/working/outputs && zip -r lora_poster.zip lora_poster

  adding: lora_poster/ (stored 0%)
  adding: lora_poster/checkpoint-400/ (stored 0%)
  adding: lora_poster/checkpoint-400/unet_lora/ (stored 0%)
  adding: lora_poster/checkpoint-400/unet_lora/adapter_config.json (deflated 53%)
  adding: lora_poster/checkpoint-400/unet_lora/README.md (deflated 65%)
  adding: lora_poster/checkpoint-400/unet_lora/adapter_model.safetensors (deflated 68%)
  adding: lora_poster/checkpoint-100/ (stored 0%)
  adding: lora_poster/checkpoint-100/unet_lora/ (stored 0%)
  adding: lora_poster/checkpoint-100/unet_lora/adapter_config.json (deflated 53%)
  adding: lora_poster/checkpoint-100/unet_lora/README.md (deflated 65%)
  adding: lora_poster/checkpoint-100/unet_lora/adapter_model.safetensors (deflated 68%)
  adding: lora_poster/val_epoch_2.png (deflated 0%)
  adding: lora_poster/val_epoch_8.png (deflated 0%)
  adding: lora_poster/val_epoch_10.png (deflated 0%)
  adding: lora_poster/checkpoint-200/ (stored 0%)
  adding: lora_poster/checkpoint-200/unet_lora/ (store